In [ ]:
!pip install pandas statsmodels

In [ ]:
# SYNTHETIC DATA GENERATOR FOR TRAINING MODEL
import pandas as pd
import numpy as np
import random as rd

def synthesize_bank_data(start_date, periods, starting_balance, paycheck_days, paycheck_amount, bills_day, bills_amount, weekdays_expense_limits, weekend_expense_limits, spending_categories, income_category="Income"):
    date_range = pd.date_range(start=start_date, periods=periods, freq='D')
    data = []
    current_balance = starting_balance

    spending_probabilities = [0.5, 0.2, 0.4, 0.5, 0.75, 0.9, 0.6] # Example probabilities

    for date in date_range:
        # Paycheck
        if date.day in paycheck_days:
            change = paycheck_amount
            current_balance += change
            data.append({'Date': date, 'Change': change, 'Category': income_category, 'Balance': current_balance})

        # Bills
        if date.day == bills_day:
            change = -bills_amount
            current_balance += change
            data.append({'Date': date, 'Change': change, 'Category': 'Bills', 'Balance': current_balance})

        # Random Spending
        day_of_week = date.weekday()
        if rd.random() < spending_probabilities[day_of_week]:
            if day_of_week < 4:  # Weekdays
                change = -rd.uniform(weekdays_expense_limits[0], weekdays_expense_limits[1])
            else:  # Weekends
                change = -rd.uniform(weekend_expense_limits[0], weekend_expense_limits[1])

            current_balance += change
            category = rd.choice(spending_categories) if spending_categories else "General"
            data.append({'Date': date, 'Change': round(change, 2), 'Category': category, 'Balance': round(current_balance, 2)})

    return pd.DataFrame(data)

# Define parameters for data synthesis
START_DATE = '2023-01-01'
PERIODS = 1000
STARTING_BALANCE = 2000.00
PAYCHECK_DAYS = (1, 15)
PAYCHECK_AMOUNT = 900.00
BILLS_DAY = 1
BILLS_AMOUNT = 850.00
WEEKDAYS_EXPENSE_LIMITS = (5.00, 40.00)
WEEKEND_EXPENSE_LIMITS = (20.00, 100.00)
SPENDING_CATEGORIES = ['Food', 'Travel', 'Shopping', 'Entertainment', 'Utilities', 'Other']
INCOME_CATEGORY = 'Income'

# Synthesize the data
synthesized_df = synthesize_bank_data(
    START_DATE,
    PERIODS,
    STARTING_BALANCE,
    PAYCHECK_DAYS,
    PAYCHECK_AMOUNT,
    BILLS_DAY,
    BILLS_AMOUNT,
    WEEKDAYS_EXPENSE_LIMITS,
    WEEKEND_EXPENSE_LIMITS,
    SPENDING_CATEGORIES,
    INCOME_CATEGORY
)

display(synthesized_df.head())

print("Synthesized data is available in the 'synthesized_df' DataFrame.")

In [ ]:
# EXAMPLE OF ARIMA MODEL TRAINED TO PREDICT SPENDING IN EACH CATEGORY

import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.arima.model import ARIMAResults
import numpy as np
import pickle
from io import StringIO

# This task: trains and saves a dictionary of models (one per category)
# inside the main function to make the code self-contained and executable.


def predict_categorical_spending(uploaded_csv_content, prediction_weeks):
    """
    Predicts weekly spending for each category based on user-uploaded transaction data.

    Args:
        uploaded_csv_content (str): The content of the user's CSV file.
        prediction_weeks (int): The number of weeks to forecast.

    Returns:
        pd.DataFrame: A DataFrame with predicted weekly spending per category.
    """
    
    # 2.1. Load and Preprocess Data
    
    # The uploaded CSV must contain 'Date', 'Change' (transaction amount), and 'Category'.
    try:
        df = pd.read_csv(StringIO(uploaded_csv_content))
    except Exception as e:
        return f"Error reading CSV: {e}"

    # Standardize column names for processing
    df.columns = ['Date', 'Change', 'Category']
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Filter for spending transactions (negative 'Change' amount) and convert to positive spending
    spending_df = df[df['Change'] < 0].copy()
    spending_df['Spending'] = -spending_df['Change']
    spending_df = spending_df.set_index('Date')
    
    # Check if there is enough data
    if spending_df.empty:
        return pd.DataFrame({"Error": ["No spending transactions found in the data."]})
    
    print("Data loaded and spending transactions isolated.")

    # 2.2. Aggregate Spending Data to Weekly Frequency
    
    # Get all unique spending categories
    categories = spending_df['Category'].unique()
    
    # Dictionary to hold the weekly spending data for each category
    weekly_spending = {}

    for category in categories:
        # Filter data for the current category
        category_data = spending_df[spending_df['Category'] == category]
        
        # Aggregate to a weekly frequency ('W'), summing the spending
        # Fill NaN values with 0.0 for weeks with no spending in that category
        weekly_series = category_data['Spending'].resample('W').sum().fillna(0.0)
        
        # We need at least 10 data points (weeks) to train a reasonable ARIMA model
        if len(weekly_series) > 10:
            weekly_spending[category] = weekly_series
        else:
            print(f"Skipping category '{category}' due to insufficient weekly data ({len(weekly_series)} weeks).")


    # 2.3. Train and Predict for Each Category
    
    forecast_results = {}
    
    # Placeholder ARIMA order (p, d, q). 
    # For real data, an auto_arima function should be used to find the best order.
    ARIMA_ORDER = (1, 1, 1)  # Simple order for demonstration
    
    for category, series in weekly_spending.items():
        try:
            # 1. Fit an ARIMA model to the weekly spending for this category
            # A ValueError is common if the data is constant/flat. We wrap this in a try/except.
            model = ARIMA(series, order=ARIMA_ORDER)
            model_fit = model.fit()
            
            # 2. Predict the number of steps equivalent to the requested weeks
            steps = prediction_weeks
            forecast = model_fit.forecast(steps=steps)
            
            # 3. Store the forecast (ensuring spending isn't predicted as negative)
            # Use np.maximum(0, forecast) to cap prediction at 0, as spending can't be negative.
            forecast_results[category] = np.maximum(0, forecast.round(2))
            
            print(f"Successfully forecasted for category: {category}")
            
        except ValueError as e:
            print(f"Could not fit ARIMA model for category '{category}'. Data may be too flat or short. Error: {e}")
            # Use the mean of past weekly spending as a simple fallback
            mean_spending = series.mean().round(2)
            forecast_results[category] = pd.Series([mean_spending] * prediction_weeks, index=pd.to_datetime(series.index[-1] + pd.Timedelta(weeks=i) for i in range(1, prediction_weeks + 1)))

    
    # 2.4. Structure and Output Results
    
    # Convert the dictionary of results (series) into a single DataFrame
    prediction_df = pd.DataFrame(forecast_results)
    prediction_df.index.name = 'Predicted Week End Date'
    
    # Rename columns to be clear
    prediction_df.columns = [f'Predicted Spending ({c})' for c in prediction_df.columns]
    
    # Add a 'Prediction Week' column (e.g., Week 1, Week 2, ...)
    prediction_df.insert(0, 'Prediction Week', range(1, len(prediction_df) + 1))
    
    return prediction_df


# --- 3. Example Usage ---

# Simulate the content of a user-uploaded CSV file
simulated_csv_data = """Date,Change,Category
2024-08-01,-35.00,Food
2024-08-01,900.00,Income
2024-08-02,-15.50,Food
2024-08-03,-80.00,Shopping
2024-08-04,-120.00,Entertainment
2024-08-05,-20.00,Food
2024-08-06,-45.00,Travel
2024-08-07,-150.00,Utilities
2024-08-08,-5.00,Food
2024-08-09,-10.00,Food
2024-08-10,-200.00,Shopping
2024-08-11,-100.00,Travel
2024-08-12,-30.00,Food
2024-08-13,-50.00,Food
2024-08-14,-75.00,Travel
2024-08-15,900.00,Income
2024-08-15,-850.00,Bills
2024-08-16,-10.00,Food
2024-08-17,-300.00,Entertainment
# ... (Need more data spanning several weeks for meaningful ARIMA results)
"""

# Let's create a more robust simulated data set for the demo to work
from datetime import timedelta, date
start_date = date(2023, 1, 1)
data_rows = []
categories_list = ['Food', 'Travel', 'Shopping', 'Entertainment', 'Utilities', 'Other']
for i in range(100): # 100 days of data
    d = start_date + timedelta(days=i)
    
    # 70% chance of a transaction
    if np.random.rand() > 0.3:
        amount = np.random.uniform(5, 100)
        category = np.random.choice(categories_list, p=[0.4, 0.15, 0.15, 0.1, 0.1, 0.1])
        data_rows.append([d, -amount, category])
    
    # Simulate a monthly rent bill (a high, fixed transaction)
    if d.day == 1:
        data_rows.append([d, -1200.00, 'Bills'])

simulated_df = pd.DataFrame(data_rows, columns=['Date', 'Change', 'Category'])
simulated_csv_content = simulated_df.to_csv(index=False)


# --- User Input Simulation ---
PREDICTION_WEEKS = 4  # The user specifies this number of weeks

# Run the prediction
predicted_spending_df = predict_categorical_spending(simulated_csv_content, PREDICTION_WEEKS)

print("\n" + "="*50)
print(f"Predicted Spending for the next {PREDICTION_WEEKS} Weeks")
print("="*50)
print(predicted_spending_df)

TODO:
- in this currnet implementation, ARIMA models need to be trianed for EACH category 
- training an ARIMA model for each category may not be the most effective in the long run, but that's what i have so far
- work to make code compatible with current program (it is currently self-contained, but we need to make it so that csv input from 
  user can be passed to it)
- ^ adding on, we need to implement the synthetic data to train the arima models for each category
- generate synthetic csv to mimic user input
- implement goal gauging system to tell user if they are on track to spend less than *blank* amount each week


May need to look at alternative ways to predict spending per category